# 🚀 ASVspoof 5 - Linguistic Transcribe (Teammate PC Version)

This notebook is designed to run **locally on a PC** with a good GPU and internet connection. 
- It avoids Google Colab commands.
- It automatically downloads only **3 specific batches** (~20GB total) to save time/space.
- It uses **openai/whisper-small**.
- It automatically deletes the heavy `.tar` and `.flac` files as it goes, so it never takes up more than ~15GB of disk space at any time.

In [ ]:
# 1. Setup Environment
!pip install zenodo_get transformers accelerate datasets soundfile pandas numpy tqdm torch

In [ ]:
# 2. Setup Local Paths
import os, shutil, tarfile, json, requests
import pandas as pd, numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import torch

BASE_DIR = Path('./ASVspoof_Local_Workspace')
LING_DIR = BASE_DIR / 'Linguistic_Dataset'
LING_DIR.mkdir(parents=True, exist_ok=True)

TEMP_DIR = BASE_DIR / 'temp_flacs'
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace Ready at: {BASE_DIR.absolute()}")

In [ ]:
# 3. Fetch Zenodo Info & Load Protocols
ZENODO_RECORD = '14498691'
API_URL = f"https://zenodo.org/api/records/{ZENODO_RECORD}"
res = requests.get(API_URL).json()
files_info = {f['key']: f['links']['self'] for f in res['files']}

PROTOCOL_TAR = 'ASVspoof5_protocols.tar.gz'
PROTOCOL_DIR = BASE_DIR / 'protocols'

if not PROTOCOL_DIR.exists():
    print("Downloading protocols from Zenodo...")
    PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
    
    import urllib.request
    tar_path = BASE_DIR / PROTOCOL_TAR
    urllib.request.urlretrieve(files_info[PROTOCOL_TAR], tar_path)
    
    with tarfile.open(tar_path, 'r:gz') as tar: tar.extractall(path=PROTOCOL_DIR)
    tar_path.unlink()

protocol_map = {}
protocol_files = [f for f in PROTOCOL_DIR.rglob('*') if f.suffix in ['.txt', '.tsv']]
print(f"Found {len(protocol_files)} protocol files. Parsing...")

for split_file in protocol_files:
    with open(split_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('#') or line.startswith('speaker_id') or not line.strip(): 
                continue
            parts = line.strip().split()
            if len(parts) >= 5:
                label = 0 if parts[4].lower() == 'bonafide' else 1
                fname = parts[1] if parts[1].endswith('.flac') else parts[1] + '.flac'
                protocol_map[fname] = label

print(f"Loaded {len(protocol_map)} entries into protocol_map.")

In [ ]:
# 4. Initialize Whisper Pipeline on GPU
from transformers import pipeline
device = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if device == 0 else 'CPU'
print(f"Initializing whisper-small on: {device_name}...")

transcriber = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-small',
    device=device,
    generate_kwargs={'language': 'english', 'task': 'transcribe'},
    chunk_length_s=30,
    torch_dtype=torch.float16 if device == 0 else torch.float32,
)

In [ ]:
# 5. Whisper Batch Transcriber Function
def transcribe_batch(file_paths):
    from datasets import Dataset, Audio
    valid_paths = [f for f in file_paths if f.name in protocol_map]
    if not valid_paths:
        return []
    
    ds = Dataset.from_dict({'audio': [str(p) for p in valid_paths], 'filename': [p.name for p in valid_paths]})
    ds = ds.cast_column('audio', Audio(sampling_rate=16000))
    
    def audio_gen(dataset):
        for sample in dataset: yield sample['audio']
        
    results = []
    for sample, out in zip(ds, transcriber(audio_gen(ds), batch_size=16)):
        results.append({
            'filename': sample['filename'],
            'text': (out.get('text') or '').strip(),
            'label': protocol_map[sample['filename']]
        })
    return results

In [ ]:
# 6. Download & Extract (Limited to 3 Batches)
def download_file(url, dest):
    import urllib.request
    class DownloadProgressBar(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None: self.total = tsize
            self.update(b * bsize - self.n)
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=dest.name) as t:
        urllib.request.urlretrieve(url, filename=dest, reporthook=t.update_to)

target_batches = ['flac_T_aa.tar', 'flac_D_aa.tar', 'flac_E_aa.tar']

for tar_name in target_batches:
    print(f"\n{'='*50}\nProcessing {tar_name}...\n{'='*50}")
    url = files_info[tar_name]
    tar_path = BASE_DIR / tar_name
    
    # Download
    if not tar_path.exists():
        print(f"Downloading {tar_name}...")
        download_file(url, tar_path)
        
    # Extract
    if TEMP_DIR.exists(): shutil.rmtree(TEMP_DIR)
    TEMP_DIR.mkdir(parents=True, exist_ok=True)
    
    print("Extracting audio files...")
    with tarfile.open(tar_path, 'r') as tar: 
        if hasattr(tarfile, 'data_filter'):
            tar.extractall(path=TEMP_DIR, filter='data')
        else:
            tar.extractall(path=TEMP_DIR)
            
    all_flacs = list(TEMP_DIR.rglob('*.flac'))
    print(f"Extracted {len(all_flacs)} audio files.")
    
    # Transcribe
    split_char = tar_name.split('_')[1]
    split_name = 'train' if split_char == 'T' else 'dev' if split_char == 'D' else 'eval'
    csv_path = LING_DIR / f"transcripts_{split_name}.csv"
    
    print("Starting GPU Transcription...")
    t_results = transcribe_batch(all_flacs)
    
    if t_results:
        print(f"Saving {len(t_results)} rows to {csv_path.name}...")
        pd.DataFrame(t_results).to_csv(csv_path, mode='a', header=not csv_path.exists(), index=False)
        
    # Cleanup to save PC Disk Space
    print("Cleaning up temporary files...")
    shutil.rmtree(TEMP_DIR)
    tar_path.unlink()

print("\n\ud83c\udf89 Dataset Extraction Complete!")
print("Please send the 3 CSV files located in 'ASVspoof_Local_Workspace/Linguistic_Dataset' to your teammate.")